### Pip install que precisam ocorrer antes de importe de blibliotecas especificas

In [ ]:

%pip install uv
!uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install xformers --index-url https://download.pytorch.org/whl/cu121

!uv pip install unsloth unsloth-zoo torchao
!uv pip install accelerate transformers trl peft bitsandbytes datasets
!uv pip install setuptools
!uv pip install pandas scikit-learn google-generativeai matplotlib ipywidgets

/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
Using Python 3.10.21 environment at: myenv
Resolved 26 packages in 3.19s                                        
Prepared 21 packages in 45.32s                                               nvidia-cublas-cu12       ------------------------------ 390.48 MiB/391.57 MiB   torch                    ------------------------------ 744.20 MiB/744.28 MiB   torch                    ------------------------------ 518.49 MiB/744.28 MiB   torch                    ------------------------------ 511.43 MiB/744.28 MiB   torch                    ------------------------------ 490.21 MiB/744.28 MiB   torch                    ------------------------------ 379.24 MiB/744.28 MiB   torch                    ------------------------------ 331.78 MiB/744.28 MiB   torch                    ------------------------------ 198.55 MiB/744.28 MiB   torch       

In [1]:
import os
import time
import torch
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import google.generativeai as genai

# Verificação de Hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Nenhuma GPU'
print(f"Iniciando Pipeline em: {device} | Dispositivo: {gpu_name}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not build the patched trl.trainer.grpo_trainer, so training will use trl's own trainer instead: RuntimeError: Direct module loading failed for UnslothGRPOTrainer: Unexpected optimization option triton.enable_persistent_tma_matmul, known options are ['TYPE_CHECKING', 'enable_auto_functionalized_v2', 'debug', 'disable_progress', 'verbose_progress', 'fx_graph_cache', 'fx_graph_remote_cache', 'autotune_local_cache', 'autotune_remote_cache', 'force_disable_caches', 'sleep_sec_TESTING_ONLY', 'custom_op_default_layout_constraint', 'cpp_wrapper', 'abi_compatible', 'c_shim_version', 'dce', 'static_weight_shapes', 'size_asserts', 'nan_asserts', 'pick_loop_orders', 'inplace_buffers', 'allow_buffer_reuse', 'memory_planning', 'memory_pool', 'benchmark_harness', 'epilogue_fusion', 'epilogue_fusion_first', 'pattern_matcher', 'b2b_gemm_pass', 'post_grad_custom_pre_pass', 'post_grad_custom_post_pass', 'joint_custom_pre_pass', 'joint_custom_post_pass', 'pre_grad_c

Iniciando Pipeline em: cuda | Dispositivo: NVIDIA GeForce RTX 3060


/tmp/ipykernel_173253/4131812714.py:9: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
import os
import google.generativeai as genai
from datasets import Dataset
import pandas as pd
import time
from sklearn.model_selection import train_test_split


def carregar_e_preparar_dados(caminho_csv: str) -> pd.DataFrame:
    # Carrega mantendo os nomes originais de colunas
    df = pd.read_csv(caminho_csv, sep=";", encoding="latin1")
    df.columns = df.columns.str.strip().str.upper()

    # Define label verdadeiro para os dados reais
    df["LABEL_CATEGORY"] = "true"
    return df


def gerar_proposta_falsa(candidato: str, proposta_real: str) -> str:
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        return ""

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemini-1.5-flash")

    prompt = f"""Você é um gerador de dados sintéticos para treinamento de checagem de fatos.
Candidato: {candidato}
Proposta Real: {proposta_real}

Gere UMA proposta FALSA/DISTORCIDA que pareça ter sido dita pelo candidato "{candidato}", baseando-se na proposta real acima.
Aplique um exagero inviável, alteração de público-alvo ou inclusão de custos/regras absurdas.
Retorne APENAS o texto da proposta falsa."""

    try:
        response = model.generate_content(prompt)
        return response.text.strip() if response.text else ""
    except Exception as e:
        print(f"Erro ao gerar fake para {candidato}: {e}")
        return ""


# 1. Carregamento da base real
df_real = carregar_e_preparar_dados("./com_propostas.csv")

# 2. Geração sintética pareada por candidato
falsas_registros = []
print("Gerando propostas falsas vinculadas aos candidatos...")

for _, row in df_real.iterrows():
    cand_nome = row.get("NM_CANDIDATO", row.get("NM_URNA_CANDIDATO", "Desconhecido"))
    prop_real = row.get("PROPOSTA", "")

    if pd.isna(prop_real) or not str(prop_real).strip():
        continue

    prop_falsa = gerar_proposta_falsa(str(cand_nome), str(prop_real))
    
    time.sleep(3)

    if prop_falsa:
        falsas_registros.append(
            {
                "DS_CARGO": row.get("DS_CARGO", ""),
                "NM_UE": row.get("NM_UE", ""),
                "SQ_CANDIDATO": row.get("SQ_CANDIDATO", ""),
                "NM_CANDIDATO": row.get("NM_CANDIDATO", cand_nome),
                "NM_URNA_CANDIDATO": row.get("NM_URNA_CANDIDATO", cand_nome),
                "PROPOSTA": prop_falsa,
                "LABEL_CATEGORY": "false",
            }
        )

df_falsas = pd.DataFrame(falsas_registros)
df_final = pd.concat([df_real, df_falsas], ignore_index=True)

# Embaralha os dados para treino
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

Gerando propostas falsas vinculadas aos candidatos...


In [ ]:
max_seq_length = 4096
lora_rank = 32          # Rank 16 otimiza o uso da GPU sem perda relevante de precisão

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype=None,
)

# Configuração LoRA/PEFT para SFT
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.9.7: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.68 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.6. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/huggingface_hub/constants.py:301: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
# Template ajustado para incluir a relação Candidato + Proposta
prompt_template = """### Instrução:
Classifique a proposta política a seguir atribuída ao candidato {candidato} como 'verdadeira' (true) ou 'falsa' (false).

### Candidato:
{candidato}

### Proposta:
{proposta}

### Resposta:
{resposta}"""


def formatar_prompts(dataframe):
    textos = []
    for _, row in dataframe.iterrows():
        candidato = row["NM_CANDIDATO"]
        if pd.isna(candidato) or not str(candidato).strip():
            candidato = row["NM_URNA_CANDIDATO"]

        texto = (
            prompt_template.format(
                candidato=candidato,
                proposta=row["PROPOSTA"],
                resposta=row["LABEL_CATEGORY"],
            )
            + tokenizer.eos_token
        )
        textos.append(texto)
    return pd.DataFrame({"text": textos})


# Divisão de Treino e Validação (80/20)
train_df, val_df = train_test_split(
    df_final, test_size=0.2, random_state=42, stratify=df_final["LABEL_CATEGORY"]
)

dataset_treino = Dataset.from_pandas(formatar_prompts(train_df))
dataset_validacao = Dataset.from_pandas(formatar_prompts(val_df))

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_treino,
    eval_dataset = dataset_validacao,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = SFTConfig(
        per_device_train_batch_size = 4,   
        gradient_accumulation_steps = 4,     
        warmup_ratio = 0.05,
        num_train_epochs = 4,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_propostas",
    ),
)

# Iniciar Treinamento SFT
trainer_stats = trainer.train()
print("Treinamento finalizado com sucesso!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: reducing dataset_num_proc 6 -> 3 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/329 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/83 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


...

Unsloth: Will smartly offload gradients to save VRAM!


In file included from /home/cesar/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/include/python3.10/Python.h:8,
                 from /tmp/tmp005aezdt/main.c:4:
/home/cesar/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/include/python3.10/pyconfig.h:1646:9: warning: ‘_POSIX_C_SOURCE’ redefined
 1646 | #define _POSIX_C_SOURCE 200809L
      |         ^~~~~~~~~~~~~~~
In file included from /usr/include/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from /home/cesar/Documents/GitHub/challenge_1_ctrl_alt_del/myenv/lib/python3.10/site-packages/triton/backends/nvidia/include/cuda.h:56,
                 from /tmp/tmp005aezdt/main.c:2:
/usr/include/features.h:319:10: note: this is the location of the previous definition
  319 | # define _POSIX_C_SOURCE        202405L
      |          ^~~~~~~~~~~~~~~
In file included from /home/cesar/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/include/python3.10/Python.h:8,
                 from 

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.428847
10,1.350797
15,1.193751
20,1.112849
25,1.178780
30,0.992409
35,1.087100
40,1.008048
45,1.079436


KeyboardInterrupt: 

In [ ]:
gguf_directory = "modelo_propostas_gguf"
quantization_method = "q4_k_m" # Opções: "q4_k_m", "q8_0", "f16"
# pra rodar localmente ou em outras plataformas
print(f"Exportando modelo para GGUF em quantização {quantization_method}...")

model.save_pretrained_gguf(
    gguf_directory, 
    tokenizer, 
    quantization_method = quantization_method
)

print(f"Modelo GGUF salvo com sucesso na pasta: ./{gguf_directory}")